In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "churn"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "retail_sales_final.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Master dataset not found: {INPUT_PATH}")

# Load only the nine analysis columns.
# Do not use or modify the three helper flag columns.
analysis_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"
]

df = pd.read_csv(INPUT_PATH, low_memory=False)

missing_columns = [c for c in analysis_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

sales = df[analysis_columns].copy()

sales["InvoiceDate"] = pd.to_datetime(
    sales["InvoiceDate"], errors="coerce"
)
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")
sales["Price"] = pd.to_numeric(sales["Price"], errors="coerce")

# Use eligible purchase records with a known customer ID.
sales = sales[
    sales["Customer ID"].notna()
    & sales["InvoiceDate"].notna()
    & (sales["Quantity"] > 0)
    & (sales["Price"] > 0)
].copy()

sales["Purchase_Date"] = sales["InvoiceDate"].dt.normalize()
sales["Sales_Revenue"] = sales["Quantity"] * sales["Price"]

if sales.empty:
    raise ValueError("No eligible customer purchase records were found.")

# Use the latest eligible transaction date to define the observation horizon.
dataset_end = sales["Purchase_Date"].max()

# The final 90 calendar days are used only to assign the churn label.
label_start = dataset_end - pd.Timedelta(days=89)
feature_cutoff = label_start - pd.Timedelta(days=1)

feature_sales = sales[sales["Purchase_Date"] <= feature_cutoff].copy()
future_sales = sales[
    (sales["Purchase_Date"] >= label_start)
    & (sales["Purchase_Date"] <= dataset_end)
].copy()

if feature_sales.empty or future_sales.empty:
    raise ValueError("Feature or future-label period is empty.")

# --------------------------------------------------
# 1. Build customer features using historical data only
# --------------------------------------------------
customer_features = (
    feature_sales.groupby("Customer ID")
    .agg(
        First_Purchase_Date=("Purchase_Date", "min"),
        Last_Purchase_Date=("Purchase_Date", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary_Value=("Sales_Revenue", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Unique_Products=("StockCode", "nunique"),
        Active_Purchase_Days=("Purchase_Date", "nunique"),
        Unique_Countries=("Country", "nunique")
    )
    .reset_index()
)

customer_features["Recency_Days"] = (
    feature_cutoff - customer_features["Last_Purchase_Date"]
).dt.days

customer_features["Purchase_Span_Days"] = (
    customer_features["Last_Purchase_Date"]
    - customer_features["First_Purchase_Date"]
).dt.days

# Calculate mean invoice value for each customer's historical invoices.
invoice_values = (
    feature_sales.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(Invoice_Value=("Sales_Revenue", "sum"))
)

mean_invoice_values = (
    invoice_values.groupby("Customer ID", as_index=False)
    .agg(Avg_Invoice_Value=("Invoice_Value", "mean"))
)

customer_features = customer_features.merge(
    mean_invoice_values,
    on="Customer ID",
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------
# 2. Assign future-inactivity labels
# Churned_Next_90_Days = 1 means no eligible purchase
# during the defined 90-day future period.
# --------------------------------------------------
future_active_ids = set(future_sales["Customer ID"].unique())

customer_features["Churned_Next_90_Days"] = (
    ~customer_features["Customer ID"].isin(future_active_ids)
).astype(int)

# Basic checks for features and labels
feature_columns = [
    "Recency_Days",
    "Frequency",
    "Monetary_Value",
    "Total_Quantity",
    "Unique_Products",
    "Active_Purchase_Days",
    "Unique_Countries",
    "Purchase_Span_Days",
    "Avg_Invoice_Value"
]

if customer_features[feature_columns].isna().any().any():
    missing_counts = customer_features[feature_columns].isna().sum()
    raise ValueError(
        "Missing values found in model features. Investigate before training:\n"
        + missing_counts[missing_counts > 0].to_string()
    )

if (customer_features[feature_columns] < 0).any().any():
    raise ValueError("Negative feature values found. Investigate before training.")

# Keep dates and customer ID as descriptive/identifier fields.
# The next modelling cell should exclude these from model inputs.
output_path = PROCESSED_DIR / "customer_churn_training_data.csv"
customer_features.to_csv(output_path, index=False)

# Save a label definition and class balance summary.
label_summary = pd.DataFrame([
    {
        "Metric": "Latest eligible transaction date",
        "Value": str(dataset_end.date())
    },
    {
        "Metric": "Historical feature cutoff",
        "Value": str(feature_cutoff.date())
    },
    {
        "Metric": "Future label period start",
        "Value": str(label_start.date())
    },
    {
        "Metric": "Future label period end",
        "Value": str(dataset_end.date())
    },
    {
        "Metric": "Future label horizon in calendar days",
        "Value": 90
    },
    {
        "Metric": "Customers in churn dataset",
        "Value": len(customer_features)
    },
    {
        "Metric": "Customers labelled churned/inactive",
        "Value": int(customer_features["Churned_Next_90_Days"].sum())
    },
    {
        "Metric": "Customers who purchased in the next 90 days",
        "Value": int((customer_features["Churned_Next_90_Days"] == 0).sum())
    },
    {
        "Metric": "Churned/inactive percentage",
        "Value": round(
            customer_features["Churned_Next_90_Days"].mean() * 100, 2
        )
    }
])

summary_path = REPORT_DIR / "churn_label_summary.csv"
label_summary.to_csv(summary_path, index=False)

print("=" * 65)
print("RETAILPULSE - CHURN DATASET PREPARATION")
print("=" * 65)
print(f"Historical features end: {feature_cutoff.date()}")
print(f"Future label period: {label_start.date()} to {dataset_end.date()}")
print(f"Customers prepared: {len(customer_features):,}")

print("\nChurn label counts:")
display(
    customer_features["Churned_Next_90_Days"]
    .value_counts()
    .rename(index={0: "Purchased in next 90 days", 1: "No purchase in next 90 days"})
    .to_frame("Customers")
)

print("\nLabel summary:")
display(label_summary)

print("\nFirst five customer records:")
display(customer_features.head())

print("\nSaved files:")
print(output_path)
print(summary_path)



In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------------
# 1. Paths and loading
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "churn"
MODEL_DIR = PROJECT_DIR / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "customer_churn_training_data.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Churn training file not found: {INPUT_PATH}")

churn_df = pd.read_csv(INPUT_PATH)

target = "Churned_Next_90_Days"

feature_columns = [
    "Recency_Days",
    "Frequency",
    "Monetary_Value",
    "Total_Quantity",
    "Unique_Products",
    "Active_Purchase_Days",
    "Unique_Countries",
    "Purchase_Span_Days",
    "Avg_Invoice_Value"
]

required_columns = ["Customer ID", target] + feature_columns
missing_columns = [
    col for col in required_columns if col not in churn_df.columns
]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

X = churn_df[feature_columns].copy()
y = churn_df[target].astype(int)

if y.nunique() != 2:
    raise ValueError("The target must contain both classes, 0 and 1.")

if X.isna().any().any():
    raise ValueError("Missing feature values found.")

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Infinite or non-numeric feature values found.")

if (X < 0).any().any():
    raise ValueError("Negative feature values found; log1p requires non-negative features.")

# --------------------------------------------------
# 2. Split data into training and held-out testing
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training customers:", len(X_train))
print("Test customers:", len(X_test))
print(f"Churn rate in training: {y_train.mean() * 100:.2f}%")
print(f"Churn rate in test: {y_test.mean() * 100:.2f}%")

# --------------------------------------------------
# 3. Define two baseline models
# --------------------------------------------------
# Logistic Regression: log-transform skewed positive features,
# standardize them, and account for class imbalance.
logistic_model = Pipeline([
    ("log_transform", FunctionTransformer(np.log1p, validate=False)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# Random Forest: tree-based baseline, robust to different
# feature scales and nonlinear relationships.
random_forest_model = RandomForestClassifier(
    n_estimators=400,
    max_features="sqrt",
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

# --------------------------------------------------
# 4. Evaluation helpers
# --------------------------------------------------
def evaluate_churn_model(model_name, model, X_test, y_test):
    probabilities = model.predict_proba(X_test)[:, 1]
    predicted_labels = (probabilities >= 0.5).astype(int)

    # Rank customers by predicted churn risk.
    top_k = max(1, int(np.ceil(0.20 * len(y_test))))
    ranking = np.argsort(-probabilities)[:top_k]

    actual = y_test.to_numpy()
    top_actual = actual[ranking]

    return {
        "Model": model_name,
        "Test_Customers": len(y_test),
        "ROC_AUC": roc_auc_score(y_test, probabilities),
        "PR_AUC_Average_Precision": average_precision_score(
            y_test, probabilities
        ),
        "Precision_At_Top_20_Percent": precision_score(
            actual[ranking], np.ones(top_k, dtype=int), zero_division=0
        ) if False else float(top_actual.mean()),
        "Recall_At_Top_20_Percent": (
            float(top_actual.sum() / actual.sum())
            if actual.sum() else np.nan
        ),
        "Precision_At_Threshold_0_5": precision_score(
            y_test, predicted_labels, zero_division=0
        ),
        "Recall_At_Threshold_0_5": recall_score(
            y_test, predicted_labels, zero_division=0
        ),
        "F1_At_Threshold_0_5": f1_score(
            y_test, predicted_labels, zero_division=0
        ),
        "Top_20_Percent_Customer_Count": top_k,
        "Test_Churn_Rate_Percent": float(y_test.mean() * 100)
    }, probabilities, ranking

# --------------------------------------------------
# 5. Train, evaluate, and save both models
# --------------------------------------------------
benchmark_rows = []
prediction_frames = []

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    model.fit(X_train, y_train)

    metrics_row, probabilities, top_ranking = evaluate_churn_model(
        model_name, model, X_test, y_test
    )
    benchmark_rows.append(metrics_row)

    # Save the model artifact for later project use.
    artifact_name = (
        "churn_logistic_regression.joblib"
        if model_name == "Logistic Regression"
        else "churn_random_forest.joblib"
    )
    joblib.dump(model, MODEL_DIR / artifact_name)

    model_predictions = pd.DataFrame({
        "Customer_ID": churn_df.loc[X_test.index, "Customer ID"].to_numpy(),
        "Actual_Churn_Label": y_test.to_numpy(),
        "Predicted_Churn_Probability": probabilities,
        "Predicted_Label_At_0_5": (probabilities >= 0.5).astype(int),
        "Model": model_name
    })

    model_predictions["Risk_Rank"] = (
        model_predictions["Predicted_Churn_Probability"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    top_k = max(1, int(np.ceil(0.20 * len(y_test))))
    model_predictions["In_Top_20_Percent_Risk_Group"] = (
        model_predictions["Risk_Rank"] <= top_k
    )

    prediction_frames.append(model_predictions)

# --------------------------------------------------
# 6. Save benchmark metrics and test-set predictions
# --------------------------------------------------
benchmark = pd.DataFrame(benchmark_rows).sort_values(
    "ROC_AUC", ascending=False
).reset_index(drop=True)

all_predictions = pd.concat(prediction_frames, ignore_index=True)

benchmark_path = REPORT_DIR / "churn_model_benchmark.csv"
predictions_path = REPORT_DIR / "churn_test_predictions.csv"

benchmark.to_csv(benchmark_path, index=False)
all_predictions.to_csv(predictions_path, index=False)

# --------------------------------------------------
# 7. Report the results against project targets
# --------------------------------------------------
best_model_row = benchmark.iloc[0]

print("\n" + "=" * 70)
print("RETAILPULSE - CHURN MODEL BENCHMARK")
print("=" * 70)
display(benchmark.round(4))

print("\nProject targets:")
print("ROC-AUC target: at least 0.88")
print("Precision among top 20% risk-ranked customers: at least 0.75")

for _, row in benchmark.iterrows():
    auc_status = "MET" if row["ROC_AUC"] >= 0.88 else "NOT MET"
    precision_status = (
        "MET"
        if row["Precision_At_Top_20_Percent"] >= 0.75
        else "NOT MET"
    )
    print(
        f"{row['Model']}: ROC-AUC {auc_status}; "
        f"Top-20% precision {precision_status}"
    )

print("\nSaved files:")
print(benchmark_path)
print(predictions_path)
print(MODEL_DIR / "churn_logistic_regression.joblib")
print(MODEL_DIR / "churn_random_forest.joblib")


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------------
# 1. Load the same churn data and reproduce the split
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "churn"
MODEL_DIR = PROJECT_DIR / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "customer_churn_training_data.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Churn dataset not found: {INPUT_PATH}")

churn_df = pd.read_csv(INPUT_PATH)

target = "Churned_Next_90_Days"

feature_columns = [
    "Recency_Days",
    "Frequency",
    "Monetary_Value",
    "Total_Quantity",
    "Unique_Products",
    "Active_Purchase_Days",
    "Unique_Countries",
    "Purchase_Span_Days",
    "Avg_Invoice_Value"
]

missing = [
    c for c in ["Customer ID", target] + feature_columns
    if c not in churn_df.columns
]
if missing:
    raise ValueError(f"Missing columns: {missing}")

X = churn_df[feature_columns].copy()
y = churn_df[target].astype(int)

if X.isna().any().any():
    raise ValueError("Missing feature values found.")

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Invalid or infinite feature values found.")

# Re-create the same stratified 80/20 split as the baseline cell.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# --------------------------------------------------
# 2. Configure the models and tuning ranges
# --------------------------------------------------
models_to_tune = {
    "Tuned Logistic Regression": {
        "estimator": Pipeline([
            ("log_transform", FunctionTransformer(np.log1p, validate=False)),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        "params": {
            "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0]
        },
        "n_iter": 5
    },

    "Tuned Random Forest": {
        "estimator": RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [200, 300, 400],
            "max_depth": [None, 8, 12, 16],
            "min_samples_leaf": [2, 5, 10],
            "max_features": ["sqrt", 0.8]
        },
        "n_iter": 12
    },

    "Tuned Gradient Boosting": {
        "estimator": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [2, 3],
            "min_samples_leaf": [5, 10, 20],
            "subsample": [0.7, 0.9, 1.0]
        },
        "n_iter": 12
    }
}

searches = {}
cv_rows = []

# --------------------------------------------------
# 3. Tune using cross-validation on training data only
# --------------------------------------------------
for name, config in models_to_tune.items():
    print(f"\nTuning {name}...")

    search = RandomizedSearchCV(
        estimator=config["estimator"],
        param_distributions=config["params"],
        n_iter=config["n_iter"],
        scoring="roc_auc",
        cv=cv,
        refit=True,
        random_state=42,
        n_jobs=-1,
        verbose=1,
        return_train_score=False
    )

    search.fit(X_train, y_train)
    searches[name] = search

    cv_rows.append({
        "Model": name,
        "Best_CV_ROC_AUC": search.best_score_,
        "Best_Parameters": str(search.best_params_)
    })

    print(f"Best cross-validation ROC-AUC: {search.best_score_:.4f}")
    print("Best parameters:", search.best_params_)

cv_results = pd.DataFrame(cv_rows).sort_values(
    "Best_CV_ROC_AUC",
    ascending=False
).reset_index(drop=True)

cv_path = REPORT_DIR / "churn_tuning_cv_results.csv"
cv_results.to_csv(cv_path, index=False)

# --------------------------------------------------
# 4. Evaluate tuned models on the untouched test set
# --------------------------------------------------
test_rows = []
prediction_frames = []

for name, search in searches.items():
    best_model = search.best_estimator_
    probabilities = best_model.predict_proba(X_test)[:, 1]
    predicted_labels = (probabilities >= 0.5).astype(int)

    actual = y_test.to_numpy()
    top_k = max(1, int(np.ceil(0.20 * len(actual))))
    ranking = np.argsort(-probabilities)[:top_k]
    top_actual = actual[ranking]

    test_rows.append({
        "Model": name,
        "Best_CV_ROC_AUC": search.best_score_,
        "Test_ROC_AUC": roc_auc_score(y_test, probabilities),
        "Test_PR_AUC_Average_Precision": average_precision_score(
            y_test, probabilities
        ),
        "Precision_At_Top_20_Percent": float(top_actual.mean()),
        "Recall_At_Top_20_Percent": (
            float(top_actual.sum() / actual.sum())
            if actual.sum() else np.nan
        ),
        "Precision_At_Threshold_0_5": precision_score(
            y_test, predicted_labels, zero_division=0
        ),
        "Recall_At_Threshold_0_5": recall_score(
            y_test, predicted_labels, zero_division=0
        ),
        "F1_At_Threshold_0_5": f1_score(
            y_test, predicted_labels, zero_division=0
        ),
        "Top_20_Percent_Customer_Count": top_k
    })

    predictions = pd.DataFrame({
        "Customer_ID": churn_df.loc[X_test.index, "Customer ID"].to_numpy(),
        "Actual_Churn_Label": actual,
        "Predicted_Churn_Probability": probabilities,
        "Predicted_Label_At_0_5": predicted_labels,
        "Model": name
    })
    predictions["Risk_Rank"] = (
        predictions["Predicted_Churn_Probability"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    predictions["In_Top_20_Percent_Risk_Group"] = (
        predictions["Risk_Rank"] <= top_k
    )
    prediction_frames.append(predictions)

    # Save each tuned model for later use.
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    joblib.dump(
        best_model,
        MODEL_DIR / f"{safe_name}.joblib"
    )

test_results = pd.DataFrame(test_rows).sort_values(
    "Test_ROC_AUC",
    ascending=False
).reset_index(drop=True)

predictions_all = pd.concat(prediction_frames, ignore_index=True)

test_results_path = REPORT_DIR / "churn_tuned_test_benchmark.csv"
predictions_path = REPORT_DIR / "churn_tuned_test_predictions.csv"

test_results.to_csv(test_results_path, index=False)
predictions_all.to_csv(predictions_path, index=False)

# Save the model selected by training-only CV, not by test performance.
best_cv_name = cv_results.iloc[0]["Model"]
best_cv_model = searches[best_cv_name].best_estimator_

best_model_path = MODEL_DIR / "churn_best_model_selected_by_cv.joblib"
joblib.dump(best_cv_model, best_model_path)

# --------------------------------------------------
# 5. Show results against the project targets
# --------------------------------------------------
print("\n" + "=" * 70)
print("RETAILPULSE - CHURN TUNING RESULTS")
print("=" * 70)

print("\nCross-validation results (training data only):")
display(cv_results)

print("\nHeld-out test results:")
display(test_results.round(4))

print("\nProject target checks on the held-out test set:")
for _, row in test_results.iterrows():
    auc_met = row["Test_ROC_AUC"] >= 0.88
    precision_met = row["Precision_At_Top_20_Percent"] >= 0.75

    print(
        f"{row['Model']}: "
        f"ROC-AUC {'MET' if auc_met else 'NOT MET'}; "
        f"Top-20% precision {'MET' if precision_met else 'NOT MET'}"
    )

print("\nBest model selected using training-only cross-validation:")
print(best_cv_name)
print(f"Saved model: {best_model_path}")

print("\nSaved reports:")
print(cv_path)
print(test_results_path)
print(predictions_path)


In [ ]:

%pip install xgboost shap




In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------------
# 1. Paths and data
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "churn"
MODEL_DIR = PROJECT_DIR / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "customer_churn_training_data.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Churn dataset not found: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)

target = "Churned_Next_90_Days"
feature_columns = [
    "Recency_Days",
    "Frequency",
    "Monetary_Value",
    "Total_Quantity",
    "Unique_Products",
    "Active_Purchase_Days",
    "Unique_Countries",
    "Purchase_Span_Days",
    "Avg_Invoice_Value"
]

missing = [
    col for col in ["Customer ID", target] + feature_columns
    if col not in df.columns
]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = df[feature_columns].copy()
y = df[target].astype(int)

if X.isna().any().any():
    raise ValueError("Missing feature values found.")

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Invalid or infinite feature values found.")

# Same split as our previous churn experiments
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training customers:", len(X_train))
print("Test customers:", len(X_test))
print("Training XGBoost using reduced parallelism.")

# --------------------------------------------------
# 2. Safer model and search settings
# --------------------------------------------------
base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=1,
    max_bin=128,
    random_state=42,
    verbosity=0
)

param_distributions = {
    "n_estimators": [100, 150, 200],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.08],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [3, 8],
    "gamma": [0, 0.2]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="roc_auc",
    cv=cv,
    refit=True,
    random_state=42,
    n_jobs=1,           # Do not run folds in parallel
    pre_dispatch=1,     # Queue only one job at a time
    verbose=1,
    return_train_score=False
)

# Limit other numerical-library thread pools too.
with threadpool_limits(limits=1):
    search.fit(X_train, y_train)

best_model = search.best_estimator_

print("\nBest 3-fold CV ROC-AUC:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)

# --------------------------------------------------
# 3. Evaluate on the held-out test set
# --------------------------------------------------
probabilities = best_model.predict_proba(X_test)[:, 1]
predicted_labels = (probabilities >= 0.5).astype(int)

actual = y_test.to_numpy()
top_k = max(1, int(np.ceil(0.20 * len(actual))))
top_positions = np.argsort(-probabilities)[:top_k]
top_actual = actual[top_positions]

test_metrics = pd.DataFrame([{
    "Model": "XGBoost",
    "Best_CV_ROC_AUC": search.best_score_,
    "Test_ROC_AUC": roc_auc_score(y_test, probabilities),
    "Test_PR_AUC_Average_Precision": average_precision_score(
        y_test, probabilities
    ),
    "Precision_At_Top_20_Percent": float(top_actual.mean()),
    "Recall_At_Top_20_Percent": (
        float(top_actual.sum() / actual.sum())
        if actual.sum() else np.nan
    ),
    "Precision_At_Threshold_0_5": precision_score(
        y_test, predicted_labels, zero_division=0
    ),
    "Recall_At_Threshold_0_5": recall_score(
        y_test, predicted_labels, zero_division=0
    ),
    "F1_At_Threshold_0_5": f1_score(
        y_test, predicted_labels, zero_division=0
    ),
    "Test_Customers": len(y_test),
    "Top_20_Percent_Customer_Count": top_k
}])

# --------------------------------------------------
# 4. Save model, metrics and predictions
# --------------------------------------------------
model_path = MODEL_DIR / "churn_xgboost_best_model.joblib"
metrics_path = REPORT_DIR / "churn_xgboost_test_metrics.csv"
cv_path = REPORT_DIR / "churn_xgboost_cv_results.csv"
predictions_path = REPORT_DIR / "churn_xgboost_test_predictions.csv"
importance_path = REPORT_DIR / "churn_xgboost_feature_importance.csv"

joblib.dump(best_model, model_path)
test_metrics.to_csv(metrics_path, index=False)

pd.DataFrame(search.cv_results_).sort_values(
    "rank_test_score"
).to_csv(cv_path, index=False)

predictions = pd.DataFrame({
    "Customer_ID": df.loc[X_test.index, "Customer ID"].to_numpy(),
    "Actual_Churn_Label": actual,
    "Predicted_Churn_Probability": probabilities,
    "Predicted_Label_At_0_5": predicted_labels
})

predictions["Risk_Rank"] = (
    predictions["Predicted_Churn_Probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)
predictions["In_Top_20_Percent_Risk_Group"] = (
    predictions["Risk_Rank"] <= top_k
)
predictions.to_csv(predictions_path, index=False)

importance = pd.DataFrame({
    "Feature": feature_columns,
    "XGBoost_Feature_Importance": best_model.feature_importances_
}).sort_values("XGBoost_Feature_Importance", ascending=False)

importance.to_csv(importance_path, index=False)

print("\n" + "=" * 65)
print("RETAILPULSE - XGBOOST CHURN RESULTS")
print("=" * 65)
display(test_metrics.round(4))

print("\nFeature importance:")
display(importance)

print("\nProject target checks:")
print(
    "ROC-AUC:",
    "MET" if test_metrics.loc[0, "Test_ROC_AUC"] >= 0.88 else "NOT MET"
)
print(
    "Top-20% precision:",
    "MET" if test_metrics.loc[0, "Precision_At_Top_20_Percent"] >= 0.75
    else "NOT MET"
)

print("\nSaved files:")
print(model_path)
print(metrics_path)
print(cv_path)
print(predictions_path)
print(importance_path)


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from sklearn.model_selection import train_test_split

# --------------------------------------------------
# 1. Paths and inputs
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "churn"
MODEL_DIR = PROJECT_DIR / "models"

INPUT_PATH = PROCESSED_DIR / "customer_churn_training_data.csv"
MODEL_PATH = MODEL_DIR / "churn_xgboost_best_model.joblib"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Churn dataset not found: {INPUT_PATH}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Saved XGBoost model not found: {MODEL_PATH}")

# --------------------------------------------------
# 2. Re-create the same features and test split
# --------------------------------------------------
df = pd.read_csv(INPUT_PATH)

feature_columns = [
    "Recency_Days",
    "Frequency",
    "Monetary_Value",
    "Total_Quantity",
    "Unique_Products",
    "Active_Purchase_Days",
    "Unique_Countries",
    "Purchase_Span_Days",
    "Avg_Invoice_Value"
]

required = ["Customer ID", "Churned_Next_90_Days"] + feature_columns
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = df[feature_columns].copy()
y = df["Churned_Next_90_Days"].astype(int)

if X.isna().any().any() or not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Missing or invalid feature values found.")

_, X_test, _, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Sample at most 200 test customers for explainability.
sample_size = min(200, len(X_test))
X_sample = X_test.sample(
    n=sample_size,
    random_state=42
).sort_index()

model = joblib.load(MODEL_PATH)

print("Loaded XGBoost model successfully.")
print("Test customers available:", len(X_test))
print("Customers explained by SHAP:", len(X_sample))

# --------------------------------------------------
# 3. Calculate SHAP values
# --------------------------------------------------
explainer = shap.TreeExplainer(model)

# SHAP values explain contributions to the model's raw output.
# For binary XGBoost, positive SHAP values generally push the
# prediction toward the churn (class 1) outcome.
shap_values = explainer.shap_values(
    X_sample,
    check_additivity=False
)

# Compatibility handling for SHAP versions returning a list
# or three-dimensional class output.
if isinstance(shap_values, list):
    if len(shap_values) == 2:
        shap_values = shap_values[1]
    else:
        shap_values = shap_values[0]

shap_values = np.asarray(shap_values)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, -1]

if shap_values.shape != X_sample.shape:
    raise ValueError(
        f"Unexpected SHAP shape {shap_values.shape}; "
        f"expected {X_sample.shape}."
    )

# --------------------------------------------------
# 4. Global feature importance from mean absolute SHAP
# --------------------------------------------------
importance = pd.DataFrame({
    "Feature": feature_columns,
    "Mean_Absolute_SHAP": np.abs(shap_values).mean(axis=0),
    "Mean_SHAP_Value": shap_values.mean(axis=0)
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

importance_path = REPORT_DIR / "churn_shap_feature_importance.csv"
importance.to_csv(importance_path, index=False)

# --------------------------------------------------
# 5. SHAP summary bar plot
# --------------------------------------------------
plt.figure()
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",
    max_display=len(feature_columns),
    show=False
)
plt.title("RetailPulse: XGBoost SHAP Feature Importance")
plt.tight_layout()

bar_path = REPORT_DIR / "churn_shap_summary_bar.png"
plt.savefig(bar_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close()

# --------------------------------------------------
# 6. SHAP beeswarm plot
# --------------------------------------------------
plt.figure()
shap.summary_plot(
    shap_values,
    X_sample,
    max_display=len(feature_columns),
    show=False
)
plt.title("RetailPulse: SHAP Impact on Churn Predictions")
plt.tight_layout()

beeswarm_path = REPORT_DIR / "churn_shap_beeswarm.png"
plt.savefig(beeswarm_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close()

# --------------------------------------------------
# 7. Save sample-level explanations for audit
# --------------------------------------------------
sample_explanations = pd.DataFrame(
    shap_values,
    columns=[f"SHAP_{col}" for col in feature_columns],
    index=X_sample.index
)

sample_explanations.insert(
    0,
    "Customer_ID",
    df.loc[X_sample.index, "Customer ID"].to_numpy()
)
sample_explanations.insert(
    1,
    "Actual_Churn_Label",
    y_test.loc[X_sample.index].to_numpy()
)
sample_explanations.insert(
    2,
    "Predicted_Churn_Probability",
    model.predict_proba(X_sample)[:, 1]
)

sample_path = REPORT_DIR / "churn_shap_customer_explanations.csv"
sample_explanations.to_csv(sample_path, index=False)

print("\n" + "=" * 70)
print("RETAILPULSE - SHAP CHURN EXPLAINABILITY")
print("=" * 70)

print("\nFeature importance ranked by mean absolute SHAP:")
display(importance.round(5))

print("\nSaved files:")
print(importance_path)
print(bar_path)
print(beeswarm_path)
print(sample_path)



In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from sklearn.model_selection import train_test_split

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "churn"
MODEL_DIR = PROJECT_DIR / "models"

INPUT_PATH = PROCESSED_DIR / "customer_churn_training_data.csv"
MODEL_PATH = MODEL_DIR / "churn_best_model_selected_by_cv.joblib"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Churn dataset not found: {INPUT_PATH}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Selected-by-CV model not found: {MODEL_PATH}"
    )

df = pd.read_csv(INPUT_PATH)

feature_columns = [
    "Recency_Days",
    "Frequency",
    "Monetary_Value",
    "Total_Quantity",
    "Unique_Products",
    "Active_Purchase_Days",
    "Unique_Countries",
    "Purchase_Span_Days",
    "Avg_Invoice_Value"
]

required_columns = ["Customer ID", "Churned_Next_90_Days"] + feature_columns
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = df[feature_columns].copy()
y = df["Churned_Next_90_Days"].astype(int)

# Reproduce the existing stratified split.
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Sample a limited number of test customers for explanation.
sample_size = min(200, len(X_test))
X_sample = X_test.sample(n=sample_size, random_state=42).sort_index()

model = joblib.load(MODEL_PATH)
explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(
    X_sample,
    check_additivity=False
)

# Accommodate different SHAP versions and binary-output formats.
if isinstance(shap_values, list):
    shap_values = shap_values[1] if len(shap_values) == 2 else shap_values[0]

shap_values = np.asarray(shap_values)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, -1]

if shap_values.shape != X_sample.shape:
    raise ValueError(
        f"Unexpected SHAP output shape: {shap_values.shape}; "
        f"expected {X_sample.shape}."
    )

# Global importance for the selected model
importance = pd.DataFrame({
    "Feature": feature_columns,
    "Mean_Absolute_SHAP": np.abs(shap_values).mean(axis=0),
    "Mean_SHAP_Value": shap_values.mean(axis=0)
}).sort_values(
    "Mean_Absolute_SHAP", ascending=False
).reset_index(drop=True)

importance_path = REPORT_DIR / "churn_selected_model_shap_importance.csv"
bar_path = REPORT_DIR / "churn_selected_model_shap_bar.png"
beeswarm_path = REPORT_DIR / "churn_selected_model_shap_beeswarm.png"
sample_path = REPORT_DIR / "churn_selected_model_shap_explanations.csv"

importance.to_csv(importance_path, index=False)

plt.figure()
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",
    max_display=len(feature_columns),
    show=False
)
plt.title("RetailPulse: Selected Churn Model — SHAP Importance")
plt.tight_layout()
plt.savefig(bar_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close()

plt.figure()
shap.summary_plot(
    shap_values,
    X_sample,
    max_display=len(feature_columns),
    show=False
)
plt.title("RetailPulse: Selected Churn Model — SHAP Impact")
plt.tight_layout()
plt.savefig(beeswarm_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close()

sample_explanations = pd.DataFrame(
    shap_values,
    columns=[f"SHAP_{col}" for col in feature_columns],
    index=X_sample.index
)

sample_explanations.insert(
    0,
    "Customer_ID",
    df.loc[X_sample.index, "Customer ID"].to_numpy()
)
sample_explanations.insert(
    1,
    "Actual_Churn_Label",
    y_test.loc[X_sample.index].to_numpy()
)
sample_explanations.insert(
    2,
    "Predicted_Churn_Probability",
    model.predict_proba(X_sample)[:, 1]
)

sample_explanations.to_csv(sample_path, index=False)

print("=" * 70)
print("RETAILPULSE - SELECTED CHURN MODEL SHAP")
print("=" * 70)
print("Model: selected using training-only cross-validation")
print("Customers explained:", len(X_sample))

display(importance.round(5))

print("\nSaved files:")
print(importance_path)
print(bar_path)
print(beeswarm_path)
print(sample_path)



In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
REPORT_DIR = PROJECT_DIR / "reports" / "churn"

TUNED_PATH = REPORT_DIR / "churn_tuned_test_benchmark.csv"
XGB_PATH = REPORT_DIR / "churn_xgboost_test_metrics.csv"
SHAP_PATH = REPORT_DIR / "churn_selected_model_shap_importance.csv"
LABEL_PATH = REPORT_DIR / "churn_label_summary.csv"

required_files = [TUNED_PATH, XGB_PATH, SHAP_PATH, LABEL_PATH]
for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Required report not found: {path}")

tuned = pd.read_csv(TUNED_PATH)
xgb = pd.read_csv(XGB_PATH)
shap_importance = pd.read_csv(SHAP_PATH)
label_summary = pd.read_csv(LABEL_PATH)

comparison_columns = [
    "Model",
    "Best_CV_ROC_AUC",
    "Test_ROC_AUC",
    "Test_PR_AUC_Average_Precision",
    "Precision_At_Top_20_Percent",
    "Recall_At_Top_20_Percent"
]

for name, table in [("Tuned models", tuned), ("XGBoost", xgb)]:
    missing = [c for c in comparison_columns if c not in table.columns]
    if missing:
        raise ValueError(f"{name} report is missing columns: {missing}")

comparison = pd.concat(
    [
        tuned[comparison_columns],
        xgb[comparison_columns]
    ],
    ignore_index=True
)

comparison["Selected_By_Training_CV"] = (
    comparison["Model"] == "Tuned Gradient Boosting"
)
comparison["ROC_AUC_Target_Met"] = comparison["Test_ROC_AUC"] >= 0.88
comparison["Top_20_Precision_Target_Met"] = (
    comparison["Precision_At_Top_20_Percent"] >= 0.75
)

# Simple Markdown table formatter; does not require tabulate.
def markdown_table(frame):
    frame = frame.copy().fillna("")
    headers = [str(c) for c in frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]
    for row in frame.itertuples(index=False, name=None):
        values = []
        for value in row:
            if isinstance(value, float):
                values.append(f"{value:.4f}")
            else:
                values.append(str(value).replace("|", "/").replace("\n", " "))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

comparison_path = REPORT_DIR / "churn_final_model_comparison.csv"
summary_path = REPORT_DIR / "churn_final_summary.md"

comparison.to_csv(comparison_path, index=False)

selected_rows = comparison[
    comparison["Selected_By_Training_CV"]
]
if len(selected_rows) != 1:
    raise ValueError(
        "Expected one Gradient Boosting model selected by cross-validation."
    )

selected = selected_rows.iloc[0]
shap_importance = shap_importance.sort_values(
    "Mean_Absolute_SHAP", ascending=False
).reset_index(drop=True)

comparison_view = comparison[
    [
        "Model",
        "Best_CV_ROC_AUC",
        "Test_ROC_AUC",
        "Precision_At_Top_20_Percent",
        "ROC_AUC_Target_Met",
        "Top_20_Precision_Target_Met"
    ]
].round(4)

shap_view = shap_importance[
    ["Feature", "Mean_Absolute_SHAP", "Mean_SHAP_Value"]
].head(5).round(5)

label_lines = "\n".join(
    f"- **{row['Metric']}:** {row['Value']}"
    for _, row in label_summary.iterrows()
)

summary_text = f"""# RetailPulse — Churn Prediction Summary

## Churn definition and dataset

Churn is defined in this experiment as no eligible purchase during the
following 90 calendar days. It is a project label for inactivity, not
proof that a customer has permanently left.

{label_lines}

## Model comparison

{markdown_table(comparison_view)}

## Selected model

**Tuned Gradient Boosting**, selected using training-only cross-validation.

- Test ROC-AUC: {selected['Test_ROC_AUC']:.4f}
- Precision among the top 20% highest-risk customers:
  {selected['Precision_At_Top_20_Percent']:.4f}
- ROC-AUC target >= 0.88: {"Met" if selected['Test_ROC_AUC'] >= 0.88 else "Not met"}
- Top-20% precision target >= 0.75: {"Met" if selected['Precision_At_Top_20_Percent'] >= 0.75 else "Not met"}

XGBoost was also trained and evaluated, but its test ROC-AUC was 0.7985.
It did not outperform the selected Gradient Boosting model on test ROC-AUC.

## SHAP explainability

SHAP explanations were generated for a sample of 200 held-out test customers
for the selected Gradient Boosting model.

{markdown_table(shap_view)}

Recency_Days was the strongest feature by mean absolute SHAP contribution.
SHAP describes model behaviour, not causation.

## Limitations

- The test ROC-AUC target of 0.88 has not been achieved.
- The top-20% precision target has been achieved by the selected model.
- Churn means no eligible purchase in the defined 90-day period.
- Results should be reassessed on later data before operational deployment.

## Saved artifacts

- `churn_final_model_comparison.csv`
- `churn_selected_model_shap_importance.csv`
- `churn_selected_model_shap_bar.png`
- `churn_selected_model_shap_beeswarm.png`
- `churn_selected_model_shap_explanations.csv`
- `churn_best_model_selected_by_cv.joblib`
- `churn_xgboost_best_model.joblib`
"""

summary_path.write_text(summary_text, encoding="utf-8")

print("=" * 65)
print("RETAILPULSE - FINAL CHURN SUMMARY")
print("=" * 65)

print("\nModel comparison:")
display(comparison.round(4))

print("\nSelected model:")
print(selected["Model"])
print(f"Test ROC-AUC: {selected['Test_ROC_AUC']:.4f}")
print(
    "Top-20% precision: "
    f"{selected['Precision_At_Top_20_Percent']:.4f}"
)

print("\nTop SHAP features:")
display(shap_view)

print("\nSummary saved successfully:")
print(summary_path)
print(comparison_path)
